<a href="https://colab.research.google.com/github/VinayaSharada/FinancialAnalyticsCourse/CreditCards/CreditCardFraud/generate_synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Card Fraud Detection - Synthetic Data Generator

This notebook generates realistic synthetic credit card transaction data for educational purposes in fraud detection machine learning.

**Course:** Masters in Finance - AI/ML Applications  
**Topic:** Credit Card Fraud Detection  
**Objective:** Create realistic synthetic data that mimics real-world credit card transaction patterns

## 📦 Setup and Installation

Install required packages for the synthetic data generation.

In [ ]:
# Install required packages
! pip install pandas numpy faker matplotlib seaborn plotly -q

# Import necessary libraries
import pandas as pd
import numpy as np
from faker import Faker
import random
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ All packages installed successfully!")

## 🏭 Credit Card Data Generator Class

The main class that generates realistic synthetic credit card transaction data.

In [ ]:
class CreditCardDataGenerator:
    """
    Generates synthetic credit card transaction data with realistic patterns.
    
    This class creates datasets that mimic real-world credit card transactions
    including normal and fraudulent patterns, suitable for machine learning
    fraud detection models.
    """
    
    def __init__(self, locale='en_IN', seed=42):
        """
        Initialize the data generator.
        
        Args:
            locale (str): Faker locale for generating realistic data
            seed (int): Random seed for reproducibility
        """
        self.fake = Faker(locale)
        self.seed = seed
        random.seed(seed)
        np.random.seed(seed)
        Faker.seed(seed)
        
        print(f"✅ Initialized CreditCardDataGenerator with locale: {locale}")
        
    def generate_merchant_categories(self):
        """Generate realistic merchant categories and their risk profiles."""
        return {
            'grocery_stores': {'risk_score': 0.1, 'avg_amount': 2500, 'std_amount': 1200},
            'gas_stations': {'risk_score': 0.15, 'avg_amount': 1800, 'std_amount': 800},
            'restaurants': {'risk_score': 0.2, 'avg_amount': 1200, 'std_amount': 600},
            'online_retail': {'risk_score': 0.3, 'avg_amount': 3500, 'std_amount': 2000},
            'atm_withdrawals': {'risk_score': 0.25, 'avg_amount': 5000, 'std_amount': 2500},
            'electronics': {'risk_score': 0.4, 'avg_amount': 15000, 'std_amount': 10000},
            'jewelry': {'risk_score': 0.6, 'avg_amount': 25000, 'std_amount': 20000},
            'travel_booking': {'risk_score': 0.35, 'avg_amount': 12000, 'std_amount': 8000},
            'medical_services': {'risk_score': 0.1, 'avg_amount': 3000, 'std_amount': 2000},
            'entertainment': {'risk_score': 0.25, 'avg_amount': 2000, 'std_amount': 1000}
        }
    
    def generate_customer_profiles(self, num_customers):
        """
        Generate customer profiles with realistic Indian demographic data.
        
        Args:
            num_customers (int): Number of customer profiles to generate
            
        Returns:
            dict: Dictionary of customer profiles
        """
        print(f"🔄 Generating {num_customers} customer profiles...")
        
        customers = {}
        cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Kolkata', 'Hyderabad', 
                 'Pune', 'Ahmedabad', 'Jaipur', 'Lucknow']
        
        for i in range(num_customers):
            customer_id = f"CUST_{i+1:06d}"
            
            # Generate age and income correlation
            age = np.random.normal(35, 12)
            age = max(18, min(80, int(age)))
            
            # Income roughly correlates with age (peak earning years)
            base_income = 300000 + (age - 25) * 15000 if age > 25 else 200000
            income = max(150000, np.random.normal(base_income, base_income * 0.3))
            
            customers[customer_id] = {
                'name': self.fake.name(),
                'age': age,
                'city': random.choice(cities),
                'annual_income': int(income),
                'credit_limit': int(income * random.uniform(2, 5)),
                'account_age_months': random.randint(6, 120),
                'risk_profile': 'low' if income > 500000 else 'medium' if income > 300000 else 'high'
            }
        
        print("✅ Customer profiles generated successfully")
        return customers
    
    def generate_transactions(self, num_transactions, fraud_rate=0.02, 
                            amount_range=(100, 50000), time_range_days=90):
        """
        Generate realistic credit card transactions.
        
        Args:
            num_transactions (int): Number of transactions to generate
            fraud_rate (float): Percentage of fraudulent transactions (0.0-1.0)
            amount_range (tuple): Min and max transaction amounts in INR
            time_range_days (int): Number of days to spread transactions across
            
        Returns:
            pandas.DataFrame: Generated transaction data
        """
        print(f"🔄 Generating {num_transactions} transactions with {fraud_rate*100:.1f}% fraud rate...")
        
        # Generate customers (10% of transaction count for realistic distribution)
        num_customers = max(100, num_transactions // 10)
        customers = self.generate_customer_profiles(num_customers)
        customer_ids = list(customers.keys())
        
        merchant_categories = self.generate_merchant_categories()
        category_names = list(merchant_categories.keys())
        
        transactions = []
        start_date = datetime.now() - timedelta(days=time_range_days)
        
        # Determine fraud transactions
        num_fraud = int(num_transactions * fraud_rate)
        fraud_indices = set(random.sample(range(num_transactions), num_fraud))
        
        for i in range(num_transactions):
            customer_id = random.choice(customer_ids)
            customer = customers[customer_id]
            category = random.choice(category_names)
            cat_info = merchant_categories[category]
            
            # Generate transaction timestamp
            random_days = random.uniform(0, time_range_days)
            transaction_time = start_date + timedelta(days=random_days)
            
            # Determine if this is a fraudulent transaction
            is_fraud = i in fraud_indices
            
            # Generate amount based on category and fraud status
            if is_fraud:
                # Fraudulent transactions tend to be higher amounts or unusual patterns
                if random.random() < 0.4:  # 40% high-value fraud
                    amount = np.random.lognormal(np.log(cat_info['avg_amount'] * 3), 0.8)
                else:  # 60% regular amount fraud (harder to detect)
                    amount = np.random.normal(cat_info['avg_amount'], cat_info['std_amount'])
            else:
                # Normal transaction
                amount = np.random.normal(cat_info['avg_amount'], cat_info['std_amount'])
            
            amount = max(amount_range[0], min(amount_range[1], abs(amount)))
            
            # Generate location (fraud might be in different city)
            if is_fraud and random.random() < 0.3:  # 30% of fraud from different city
                cities = ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Kolkata']
                transaction_city = random.choice([c for c in cities if c != customer['city']])
            else:
                transaction_city = customer['city']
            
            # Generate time-based features
            hour = transaction_time.hour
            is_weekend = transaction_time.weekday() >= 5
            
            # Fraud patterns: unusual hours, weekends
            if is_fraud and random.random() < 0.4:
                hour = random.choice([2, 3, 4, 5, 22, 23])  # Unusual hours
            
            # Calculate derived features
            amount_to_income_ratio = amount / customer['annual_income'] * 12
            amount_to_limit_ratio = amount / customer['credit_limit']
            
            transaction = {
                'transaction_id': f"TXN_{i+1:08d}",
                'customer_id': customer_id,
                'timestamp': transaction_time.isoformat(),
                'amount': round(amount, 2),
                'merchant_category': category,
                'transaction_city': transaction_city,
                'customer_city': customer['city'],
                'hour_of_day': hour,
                'day_of_week': transaction_time.weekday(),
                'is_weekend': is_weekend,
                'customer_age': customer['age'],
                'customer_income': customer['annual_income'],
                'credit_limit': customer['credit_limit'],
                'account_age_months': customer['account_age_months'],
                'amount_to_income_ratio': amount_to_income_ratio,
                'amount_to_limit_ratio': amount_to_limit_ratio,
                'city_mismatch': transaction_city != customer['city'],
                'is_fraud': 1 if is_fraud else 0
            }
            
            transactions.append(transaction)
        
        df = pd.DataFrame(transactions)
        
        # Add some PCA-like anonymized features (V1-V10) similar to real fraud datasets
        np.random.seed(self.seed)
        for i in range(1, 11):
            if i <= 3:  # First few components capture most variance
                df[f'V{i}'] = np.random.normal(0, 2, len(df)) + df['is_fraud'] * np.random.normal(0, 1, len(df))
            else:
                df[f'V{i}'] = np.random.normal(0, 1, len(df))
        
        print(f"✅ Generated {len(df)} transactions ({df['is_fraud'].sum()} fraudulent)")
        return df

print("✅ CreditCardDataGenerator class loaded successfully!")

## ⚙️ Configuration Parameters

Customize the synthetic data generation parameters below:

In [ ]:
# 🎛️ CONFIGURATION PARAMETERS - Modify these to experiment with different settings

# Dataset size
NUM_TRANSACTIONS = 10000  # Number of transactions to generate

# Fraud characteristics
FRAUD_RATE = 0.02  # 2% fraud rate (adjust between 0.01-0.05 for realism)

# Amount range (in INR)
MIN_AMOUNT = 100
MAX_AMOUNT = 50000

# Time range
TIME_RANGE_DAYS = 90  # Spread transactions over 90 days

# Random seed for reproducibility
RANDOM_SEED = 42

# Output filename
OUTPUT_FILENAME = 'credit_card_transactions.csv'

print("📋 Configuration:")
print(f"   • Transactions: {NUM_TRANSACTIONS:,}")
print(f"   • Fraud Rate: {FRAUD_RATE*100:.1f}%")
print(f"   • Amount Range: ₹{MIN_AMOUNT:,} - ₹{MAX_AMOUNT:,}")
print(f"   • Time Range: {TIME_RANGE_DAYS} days")
print(f"   • Output File: {OUTPUT_FILENAME}")

## 🚀 Generate Synthetic Data

Generate the synthetic credit card transaction dataset:

In [ ]:
# Initialize the data generator
generator = CreditCardDataGenerator(locale='en_IN', seed=RANDOM_SEED)

# Generate transactions
df = generator.generate_transactions(
    num_transactions=NUM_TRANSACTIONS,
    fraud_rate=FRAUD_RATE,
    amount_range=(MIN_AMOUNT, MAX_AMOUNT),
    time_range_days=TIME_RANGE_DAYS
)

# Display basic information
print("\n" + "="*60)
print("📊 DATASET SUMMARY")
print("="*60)
print(f"Dataset shape: {df.shape}")
print(f"Total transactions: {len(df):,}")
print(f"Fraudulent transactions: {df['is_fraud'].sum():,}")
print(f"Fraud rate: {df['is_fraud'].mean()*100:.3f}%")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Amount range: ₹{df['amount'].min():.2f} to ₹{df['amount'].max():.2f}")
print(f"Average amount: ₹{df['amount'].mean():.2f}")

# Display first few rows
print("\n📋 Sample Data:")
display(df.head())

## 📈 Data Analysis and Visualization

Analyze the generated synthetic data to ensure it follows realistic patterns:

In [ ]:
# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

# Create comprehensive analysis plots
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Synthetic Credit Card Data - Quality Analysis', fontsize=16, fontweight='bold')

# 1. Fraud Distribution
fraud_counts = df['is_fraud'].value_counts()
colors = ['lightblue', 'red']
axes[0,0].pie(fraud_counts.values, labels=['Normal', 'Fraud'], autopct='%1.2f%%', 
             colors=colors, startangle=90)
axes[0,0].set_title('Transaction Distribution', fontweight='bold')

# 2. Amount Distribution by Fraud Status
df.boxplot(column='amount', by='is_fraud', ax=axes[0,1])
axes[0,1].set_title('Amount Distribution by Fraud Status', fontweight='bold')
axes[0,1].set_xlabel('Fraud Status (0=Normal, 1=Fraud)')
axes[0,1].set_ylabel('Amount (₹)')

# 3. Fraud by Hour of Day
fraud_by_hour = df.groupby('hour_of_day')['is_fraud'].mean()
axes[0,2].bar(fraud_by_hour.index, fraud_by_hour.values, color='orange', alpha=0.7)
axes[0,2].set_title('Fraud Rate by Hour of Day', fontweight='bold')
axes[0,2].set_xlabel('Hour')
axes[0,2].set_ylabel('Fraud Rate')
axes[0,2].grid(True, alpha=0.3)

# 4. Fraud by Merchant Category
fraud_by_category = df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=True)
axes[1,0].barh(fraud_by_category.index, fraud_by_category.values, color='green', alpha=0.7)
axes[1,0].set_title('Fraud Rate by Merchant Category', fontweight='bold')
axes[1,0].set_xlabel('Fraud Rate')

# 5. Amount vs Age Scatter (colored by fraud)
normal_sample = df[df['is_fraud'] == 0].sample(n=min(1000, len(df[df['is_fraud'] == 0])))
fraud_data = df[df['is_fraud'] == 1]

axes[1,1].scatter(normal_sample['customer_age'], normal_sample['amount'], 
                 alpha=0.6, label='Normal', s=10, color='blue')
axes[1,1].scatter(fraud_data['customer_age'], fraud_data['amount'], 
                 alpha=0.8, label='Fraud', s=15, color='red')
axes[1,1].set_xlabel('Customer Age')
axes[1,1].set_ylabel('Amount (₹)')
axes[1,1].set_title('Amount vs Age Distribution', fontweight='bold')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# 6. Transaction Volume by Day
df['date'] = pd.to_datetime(df['timestamp']).dt.date
daily_counts = df['date'].value_counts().sort_index()
axes[1,2].plot(daily_counts.index, daily_counts.values, color='purple', linewidth=2)
axes[1,2].set_title('Daily Transaction Volume', fontweight='bold')
axes[1,2].set_xlabel('Date')
axes[1,2].set_ylabel('Transaction Count')
axes[1,2].tick_params(axis='x', rotation=45)
axes[1,2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistical summary
print("\n📊 Statistical Summary:")
print("\nTop 5 Merchant Categories by Transaction Count:")
print(df['merchant_category'].value_counts().head())

print("\nTop 5 Cities by Transaction Count:")
print(df['transaction_city'].value_counts().head())

print("\nAmount Statistics by Fraud Status:")
print(df.groupby('is_fraud')['amount'].describe())

## 📊 Interactive Visualizations

Create interactive plots using Plotly for better data exploration:

In [ ]:
# Interactive scatter plot: Amount vs Age colored by fraud status
fig_scatter = px.scatter(df.sample(n=min(2000, len(df))), 
                        x='customer_age', y='amount', 
                        color='is_fraud',
                        color_discrete_map={0: 'blue', 1: 'red'},
                        title='Credit Card Transactions: Amount vs Customer Age',
                        labels={'is_fraud': 'Fraud Status', 
                               'customer_age': 'Customer Age',
                               'amount': 'Transaction Amount (₹)'},
                        hover_data=['merchant_category', 'transaction_city'])

fig_scatter.update_layout(height=500)
fig_scatter.show()

# Interactive time series: Daily fraud rate
daily_fraud_stats = df.groupby(df['date']).agg({
    'is_fraud': ['count', 'sum', 'mean'],
    'amount': 'mean'
}).round(4)

daily_fraud_stats.columns = ['total_transactions', 'fraud_count', 'fraud_rate', 'avg_amount']
daily_fraud_stats = daily_fraud_stats.reset_index()

fig_time = go.Figure()

# Add fraud rate line
fig_time.add_trace(go.Scatter(
    x=daily_fraud_stats['date'],
    y=daily_fraud_stats['fraud_rate'],
    mode='lines+markers',
    name='Daily Fraud Rate',
    line=dict(color='red', width=2),
    hovertemplate='Date: %{x}<br>Fraud Rate: %{y:.3f}<br>'
))

fig_time.update_layout(
    title='Daily Fraud Rate Over Time',
    xaxis_title='Date',
    yaxis_title='Fraud Rate',
    height=400,
    hovermode='x unified'
)

fig_time.show()

# Interactive bar chart: Fraud rate by merchant category
fraud_by_merchant = df.groupby('merchant_category').agg({
    'is_fraud': ['count', 'sum', 'mean']
}).round(4)

fraud_by_merchant.columns = ['total_transactions', 'fraud_count', 'fraud_rate']
fraud_by_merchant = fraud_by_merchant.reset_index().sort_values('fraud_rate', ascending=True)

fig_merchant = px.bar(fraud_by_merchant, 
                     x='fraud_rate', 
                     y='merchant_category',
                     orientation='h',
                     title='Fraud Rate by Merchant Category',
                     labels={'fraud_rate': 'Fraud Rate', 
                            'merchant_category': 'Merchant Category'},
                     color='fraud_rate',
                     color_continuous_scale='Reds')

fig_merchant.update_layout(height=500)
fig_merchant.show()

print("✅ Interactive visualizations created successfully!")

## 💾 Save Generated Data

Save the synthetic dataset to CSV format:

In [ ]:
# Save the dataset to CSV
try:
    df.to_csv(OUTPUT_FILENAME, index=False)
    print(f"✅ Dataset saved successfully to: {OUTPUT_FILENAME}")
    print(f"📁 File size: {os.path.getsize(OUTPUT_FILENAME) / (1024*1024):.2f} MB")
    
    # Display final summary
    print("\n" + "="*70)
    print("🎉 SYNTHETIC DATA GENERATION COMPLETE")
    print("="*70)
    print(f"📊 Dataset Statistics:")
    print(f"   • Total Transactions: {len(df):,}")
    print(f"   • Fraudulent Transactions: {df['is_fraud'].sum():,}")
    print(f"   • Fraud Rate: {df['is_fraud'].mean()*100:.3f}%")
    print(f"   • Date Range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"   • Amount Range: ₹{df['amount'].min():.2f} to ₹{df['amount'].max():.2f}")
    print(f"   • Average Amount: ₹{df['amount'].mean():.2f}")
    print(f"   • Unique Customers: {df['customer_id'].nunique():,}")
    print(f"   • Merchant Categories: {df['merchant_category'].nunique()}")
    print(f"   • Cities: {df['transaction_city'].nunique()}")
    
    print(f"\n📁 Output File: {OUTPUT_FILENAME}")
    print(f"\n🔄 Next Steps:")
    print(f"   1. Use this dataset with the fraud detection model")
    print(f"   2. Experiment with different parameters to generate varied datasets")
    print(f"   3. Analyze the patterns in your generated data")
    
except Exception as e:
    print(f"❌ Error saving dataset: {str(e)}")

# Show final data types and info
print("\n📋 Dataset Information:")
print(df.info())

## 🧪 Experiment with Different Parameters

Try different configurations to see how they affect the data:

In [ ]:
# 🧪 EXPERIMENT SECTION - Try different parameters

print("🧪 EXPERIMENTAL CONFIGURATIONS")
print("=" * 50)

# Experiment 1: High fraud rate scenario
print("\n🔴 Experiment 1: High Fraud Rate Scenario (5%)")
df_high_fraud = generator.generate_transactions(
    num_transactions=1000,
    fraud_rate=0.05,  # 5% fraud rate
    amount_range=(100, 50000),
    time_range_days=30
)
print(f"Fraud transactions: {df_high_fraud['is_fraud'].sum()} ({df_high_fraud['is_fraud'].mean()*100:.1f}%)")

# Experiment 2: Low fraud rate scenario  
print("\n🟡 Experiment 2: Low Fraud Rate Scenario (0.5%)")
df_low_fraud = generator.generate_transactions(
    num_transactions=1000,
    fraud_rate=0.005,  # 0.5% fraud rate
    amount_range=(100, 50000),
    time_range_days=30
)
print(f"Fraud transactions: {df_low_fraud['is_fraud'].sum()} ({df_low_fraud['is_fraud'].mean()*100:.1f}%)")

# Experiment 3: High-value transactions
print("\n💰 Experiment 3: High-Value Transactions")
df_high_value = generator.generate_transactions(
    num_transactions=1000,
    fraud_rate=0.02,
    amount_range=(10000, 100000),  # Higher amounts
    time_range_days=30
)
print(f"Average amount: ₹{df_high_value['amount'].mean():.2f}")
print(f"Max amount: ₹{df_high_value['amount'].max():.2f}")

# Compare fraud rates across experiments
print("\n📊 EXPERIMENT COMPARISON")
print("-" * 50)
print(f"{'Experiment':<25} {'Fraud Rate':<12} {'Avg Amount':<15}")
print("-" * 50)
print(f"{'Original Dataset':<25} {df['is_fraud'].mean()*100:<11.2f}% ₹{df['amount'].mean():<14.2f}")
print(f"{'High Fraud Rate':<25} {df_high_fraud['is_fraud'].mean()*100:<11.2f}% ₹{df_high_fraud['amount'].mean():<14.2f}")
print(f"{'Low Fraud Rate':<25} {df_low_fraud['is_fraud'].mean()*100:<11.2f}% ₹{df_low_fraud['amount'].mean():<14.2f}")
print(f"{'High Value':<25} {df_high_value['is_fraud'].mean()*100:<11.2f}% ₹{df_high_value['amount'].mean():<14.2f}")

print("\n💡 Key Insights:")
print("   • Higher fraud rates create more challenging detection scenarios")
print("   • Transaction amounts affect fraud patterns and detection difficulty")
print("   • Different parameters help simulate various real-world scenarios")
print("   • Use these experiments to test model robustness")

## 📚 Instructions for Students and Banks

### 🎓 For Students:

1. **Experiment with Parameters:**
   - Change `FRAUD_RATE` to see how it affects model performance
   - Modify `MIN_AMOUNT` and `MAX_AMOUNT` to simulate different transaction profiles
   - Adjust `TIME_RANGE_DAYS` to create datasets spanning different periods

2. **Analyze the Data:**
   - Study the relationship between merchant categories and fraud rates
   - Examine temporal patterns in fraudulent transactions
   - Investigate customer demographics and fraud correlation

3. **Advanced Explorations:**
   - Create datasets with seasonal patterns
   - Generate data with specific fraud scenarios (e.g., account takeover, card skimming)
   - Compare model performance across different synthetic datasets

### 🏦 For Banks:

1. **Model Training:**
   - Use synthetic data to supplement real data when privacy constraints exist
   - Create balanced datasets for training fraud detection models
   - Test model robustness across different fraud scenarios

2. **Risk Assessment:**
   - Analyze merchant category risk profiles
   - Identify high-risk transaction patterns
   - Develop targeted fraud prevention strategies

3. **Operational Benefits:**
   - Train staff using realistic but synthetic data
   - Test fraud detection systems without exposing real customer data
   - Develop and validate fraud rules and thresholds

### ⚠️ Important Notes:

- This synthetic data is for educational and testing purposes only
- Real-world fraud patterns may be more complex
- Always validate models on real data before production deployment
- Consider privacy and compliance requirements when using any transaction data